In [70]:
# This code is heavily inspired by https://github.com/gingsmith/fmtl

In [71]:
import numpy as np
from scipy.linalg import sqrtm, inv
from data import CaseHFL_Heterogenous
from sklearn.preprocessing import StandardScaler

In [72]:
def compute_rmse(X, Y, W):
    errors = []
    for t in range(len(X)):
        preds = np.dot(X[t], W[:, t])
        errors.append(np.sqrt(np.mean((Y[t] - preds)**2)))
    return np.mean(errors)

In [73]:
def compute_primal(X_train, Y_train, W, Omega, lam):
    m = len(X_train)
    total_loss = 0
    for t in range(m):
        preds = np.dot(X_train[t], W[:, t])
        total_loss += np.mean((Y_train[t] - preds)**2)
    
    reg_term = (lam / 2.0) * np.trace(W @ Omega @ W.T)
    
    return (total_loss / m) + reg_term

def compute_dual(alpha, Ytrain, W, Omega, lam):
    m = len(alpha)
    dual_loss_term = 0
    for t in range(m):
        alpha_t = alpha[t]
        y_t = Ytrain[t]
        
        dual_loss_term += np.mean(0.5 * alpha_t**2 + alpha_t * y_t)
        
    reg_dual = (lam / 2.0) * np.trace(W @ Omega @ W.T)
    
    return -(reg_dual + (dual_loss_term / m))

In [74]:
def run_mocha(X_train, Y_train, X_validation, Y_validation, lam, opts):
    num_tasks = len(X_train) 
    num_features = X_train[0].shape[1] 
    
    W = np.zeros((num_features, num_tasks))
    alpha = [np.zeros(Y_train[t].shape[0]) for t in range(num_tasks)]
    Sigma = np.eye(num_tasks) * (1.0 / num_tasks)
    Omega = inv(Sigma)
    
    num_data_points = np.array([len(Y_train[t]) for t in range(num_tasks)])
    spectral_scaling_param = 1.0
    
    num_metrics = opts['mocha_inner_iters'] if opts.get('w_update') else opts['mocha_outer_iters']
    rmse_history = np.zeros(num_metrics)
    primal_objs = np.zeros(num_metrics)
    dual_objs = np.zeros(num_metrics)

    for h in range(opts['mocha_outer_iters']):
        if not opts.get('w_update'):
            rmse_history[h] = compute_rmse(X_validation, Y_validation, W)
            primal_objs[h] = compute_primal(X_train, Y_train, W, Omega, lam)
            dual_objs[h] = compute_dual(alpha, Y_train, W, Omega, lam)

        # Update weights
        for hh in range(opts['mocha_inner_iters']):
            if opts.get('systems_heterogeneity'):
                sys_iters = (opts['top'] - opts['bottom']) * np.random.rand(num_tasks) + opts['bottom']
            
            if opts.get('w_update'):
                rmse_history[hh] = compute_rmse(X_validation, Y_validation, W)
                primal_objs[hh] = compute_primal(X_train, Y_train, W, Omega, lam)
                dual_objs[hh] = compute_dual(alpha, Y_train, W, Omega, lam)

            deltaW = np.zeros((num_features, num_tasks))
            deltaB = np.zeros((num_features, num_tasks))

            # Inner Loop: Task-parallel SDCA updates
            for task in range(num_tasks):
                alpha_t = alpha[task]
                curr_sig = Sigma[task, task]
                
                if opts.get('systems_heterogeneity'):
                    local_iters = int(num_data_points[task] * sys_iters[task])
                else:
                    local_iters = int(num_data_points[task] * opts['mocha_sdca_frac'])
                
                task_permutation = np.random.permutation(num_data_points[task])
                
                for s in range(local_iters):
                    idx = task_permutation[s % num_data_points[task]]
                    alpha_old = alpha_t[idx]
                    curr_y = Y_train[task][idx]
                    curr_x = X_train[task][idx, :]

                    # SDCA update step
                    update = curr_y * np.dot(curr_x, (W[:, task] + spectral_scaling_param * deltaW[:, task]))
                    grad = (lam * num_data_points[task] * (1.0 - update) / (curr_sig * spectral_scaling_param * np.dot(curr_x, curr_x))) + (alpha_old * curr_y)
                    
                    alpha_t[idx] = curr_y * np.clip(grad, 0.0, 1.0)
                    
                    # Update delta vectors
                    diff = alpha_t[idx] - alpha_old
                    deltaW[:, task] += Sigma[task, task] * diff * curr_x / (lam * num_data_points[task])
                    deltaB[:, task] += diff * curr_x / num_data_points[task]
                
                alpha[task] = alpha_t

            for task in range(num_tasks):
                W[:, task] += np.dot(deltaB, Sigma[task, :]) * (1.0 / lam)

        A = np.dot(W.T, W)
        
        # Ensure Positive Semi-Definiten
        eigvals, eigvecs = np.linalg.eigh(A)
        if np.any(eigvals < 0):
            eigvals[eigvals <= 1e-7] = 1e-7
            A = eigvecs @ np.diag(eigvals) @ eigvecs.T
        
        sqm = sqrtm(A)
        Sigma = sqm / np.trace(sqm)
        Omega = inv(Sigma)
        
        spectral_scaling_param = np.max(np.sum(np.abs(Sigma), axis=1) / np.diag(Sigma))

    return W, rmse_history, primal_objs, dual_objs


In [75]:
def evaluate_mocha(X_test, Y_test, W):
    num_tasks = len(X_test)
    task_mses = []
    
    for t in range(num_tasks):
        x_task = X_test[t]
        y_true = Y_test[t]
        
        w_task = W[:, t]
        
        y_pred = np.dot(x_task, w_task)
        
        mse = np.mean((y_true - y_pred)**2)
        task_mses.append(mse)
    
    mean_mse = np.mean(task_mses)
    
    return mean_mse, task_mses

In [76]:
rng = np.random.default_rng(42)
num_sensors = 50
num_events = 2000 

case = CaseHFL_Heterogenous(
    rng = rng,
    n = num_sensors * num_events,
    events = num_events,
    output_arity = 50,
    sensors = num_sensors,
    noise_std = 0.01,
    heterogeneity_std = 1.0
)

In [77]:
X_train, Y_train = case.generate_data()
X_validation, Y_validation = case.generate_data()
X_test, Y_test = case.generate_data()
scaler_x = StandardScaler()
scaler_y = StandardScaler()


X_train = [scaler_x.fit_transform(x.T).T for x in X_train]
X_validation = [scaler_x.fit_transform(x.T).T for x in X_validation]
X_test = [scaler_x.fit_transform(x.T).T for x in X_test]
Y_train = [scaler_y.fit_transform(y.reshape(-1, 1)).flatten() for y in Y_train]
Y_validation = [scaler_y.fit_transform(y.reshape(-1, 1)).flatten() for y in Y_validation]
Y_test = [scaler_y.fit_transform(y.reshape(-1, 1)).flatten() for y in Y_test]

In [78]:
opts = {
    'mocha_outer_iters': 10,   # Alternating rounds between W and Omega updates
    'mocha_inner_iters': 5,    # Number of local training rounds before central aggregation
    'mocha_sdca_frac': 0.5,    # Percentage of data processed locally in each inner round
    'systems_heterogeneity': False,    # Toggle to simulate systems heterogeneity (stragglers)
    'w_update': True,          # Track metrics during inner weight updates
    'top': 1.0,                # Max fraction for system heterogeneity simulation
    'bottom': 0.1              # Min fraction for system heterogeneity simulation
}

lambda_param = 0.01  

W, rmse, primal_objs, dual_objs = run_mocha(
    X_train, Y_train, 
    X_validation, Y_validation, 
    lambda_param, 
    opts
)

print(f"Final Average RMSE across tasks: {rmse[-1]:.4f}")

Final Average RMSE across tasks: 1.5495


In [79]:
mean_mse, task_mses = evaluate_mocha(X_test, Y_test, W)
print(mean_mse)

2.393407425093779
